# RSNA Knee DICOM Viewer

Custom DICOM viewer for the RSNA Knee Abnormality Detection competition, built to run
directly against Kaggle's own mount of the competition data (`/kaggle/input/rsna-knee-abnormality-detection`)
— no bulk download of the ~4407-study corpus required.

Features beyond a standard DICOM viewer:
- 12 radio-button controls for the competition's 12 abnormality labels, pre-filled from ground truth when available
- Original-language competition report + on-demand English translation
- On-demand `/mri-interpreter`-style AI structured report, cached per study

**Setup on Kaggle:**
1. Attach this competition as a data source to this notebook (it should already be, if forked from a competition notebook).
2. Enable internet (Settings → Internet → On) — needed for translation and AI report generation.
3. Add a Kaggle Secret named `ANTHROPIC_API_KEY` (Add-ons → Secrets) if you want AI report generation.
4. Run all cells. The viewer UI appears at the bottom.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))  # so viewer/ and rag/ import when run from repo root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from viewer.dicom_utils import find_series_for_study, load_slice_array, window_image, slice_metadata, DEFAULT_DATA_ROOT
from viewer.translate import get_or_translate
from viewer.ai_report import get_or_generate_report

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

print("Data root:", DEFAULT_DATA_ROOT)


In [ ]:
# Study list: full train.csv (competition-provided) if available on this Kaggle mount,
# else fall back to the bundled pilot list in data/pilot_studies.csv for local/offline dev.
train_csv_path = os.path.join(DEFAULT_DATA_ROOT, "train.csv")
if os.path.exists(train_csv_path):
    studies_df = pd.read_csv(train_csv_path)
    print(f"Loaded full competition train.csv: {len(studies_df)} studies")
else:
    studies_df = pd.read_csv("data/pilot_studies.csv")
    print(f"Kaggle competition data not mounted here — using bundled pilot list: {len(studies_df)} studies")

studies_df = studies_df.set_index("StudyInstanceUID", drop=False)
study_options = list(studies_df.index)


In [ ]:
# --- Widget state ---
study_dropdown = widgets.Dropdown(options=study_options, description="Study:", layout=widgets.Layout(width="600px"))
series_dropdown = widgets.Dropdown(options=[], description="Series:", layout=widgets.Layout(width="600px"))
slice_slider = widgets.IntSlider(min=0, max=0, step=1, description="Slice:", continuous_update=True)
window_slider = widgets.FloatRangeSlider(min=0, max=1, step=0.01, value=[0.005, 0.995], description="Window (pct):",
                                         layout=widgets.Layout(width="500px"))
zoom_slider = widgets.FloatSlider(min=1.0, max=4.0, step=0.25, value=1.0, description="Zoom:")

label_widgets = {lbl: widgets.RadioButtons(options=[("Absent (0)", 0), ("Present (1)", 1), ("Unlabeled", -1)],
                                            value=-1, description=lbl, style={"description_width": "140px"})
                  for lbl in LABELS}

translate_btn = widgets.Button(description="Translate report to English")
generate_report_btn = widgets.Button(description="Generate AI report (mri-interpreter)", button_style="info")
regenerate_report_btn = widgets.Button(description="Regenerate (overwrite cache)", button_style="warning")

report_original_out = widgets.Output()
report_translated_out = widgets.Output()
ai_report_out = widgets.Output()
image_out = widgets.Output()
meta_out = widgets.Output()

current_series_list = []


In [ ]:
def refresh_labels(study_uid):
    row = studies_df.loc[study_uid]
    for lbl in LABELS:
        val = row.get(lbl, None)
        if pd.isna(val):
            label_widgets[lbl].value = -1
        else:
            label_widgets[lbl].value = int(val)

def refresh_report(study_uid):
    row = studies_df.loc[study_uid]
    report_original_out.clear_output()
    with report_original_out:
        print(row.get("Report", "(no report text available)"))
    report_translated_out.clear_output()
    ai_report_out.clear_output()

def on_study_change(change):
    study_uid = change["new"]
    global current_series_list
    try:
        current_series_list = find_series_for_study(study_uid)
    except FileNotFoundError as e:
        current_series_list = []
        with image_out:
            clear_output()
            print(str(e))
        return
    series_dropdown.options = [(f"{s.plane} — {s.sequence_desc} ({len(s.slice_paths)} slices)", i)
                                for i, s in enumerate(current_series_list)]
    if current_series_list:
        series_dropdown.index = 0
    refresh_labels(study_uid)
    refresh_report(study_uid)

def on_series_change(change):
    idx = change["new"]
    if idx is None or not current_series_list:
        return
    n = len(current_series_list[idx].slice_paths)
    slice_slider.max = max(n - 1, 0)
    slice_slider.value = n // 2
    render()

def render(*_):
    image_out.clear_output(wait=True)
    meta_out.clear_output(wait=True)
    if not current_series_list or series_dropdown.value is None:
        return
    series = current_series_list[series_dropdown.value]
    if not series.slice_paths:
        return
    idx = min(slice_slider.value, len(series.slice_paths) - 1)
    path = series.slice_paths[idx]
    arr = load_slice_array(path)
    lo_pct, hi_pct = window_slider.value
    lo, hi = np.percentile(arr, lo_pct * 100), np.percentile(arr, hi_pct * 100)
    img = np.clip(arr, lo, hi)
    img = (img - lo) / (hi - lo + 1e-6)

    with image_out:
        fig_size = 5 * zoom_slider.value
        fig, ax = plt.subplots(figsize=(fig_size, fig_size))
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{series.plane} — instance {idx+1}/{len(series.slice_paths)}")
        ax.axis("off")
        plt.show()

    with meta_out:
        for k, v in slice_metadata(path).items():
            print(f"{k}: {v}")

study_dropdown.observe(on_study_change, names="value")
series_dropdown.observe(on_series_change, names="value")
slice_slider.observe(render, names="value")
window_slider.observe(render, names="value")
zoom_slider.observe(render, names="value")


In [ ]:
def on_translate_click(_):
    study_uid = study_dropdown.value
    row = studies_df.loc[study_uid]
    original = row.get("Report", "")
    report_translated_out.clear_output()
    with report_translated_out:
        print("Translating...")
    translated = get_or_translate(study_uid, str(original))
    report_translated_out.clear_output()
    with report_translated_out:
        print(translated)

def _representative_slices(max_per_series=4):
    paths, labels = [], []
    for s in current_series_list:
        n = len(s.slice_paths)
        if n == 0:
            continue
        step = max(n // max_per_series, 1)
        for i in range(0, n, step)[:max_per_series]:
            paths.append(s.slice_paths[i])
            labels.append(f"{s.plane}_{i+1}")
    return paths, labels

def _generate(force):
    study_uid = study_dropdown.value
    ai_report_out.clear_output()
    with ai_report_out:
        print("Generating..." if force else "Generating (first view — will be cached)...")
    paths, labels = _representative_slices()
    if not paths:
        with ai_report_out:
            clear_output()
            print("No slices loaded for this study — select a study with data mounted first.")
        return
    report = get_or_generate_report(study_uid, paths, labels, force_regenerate=force)
    ai_report_out.clear_output()
    with ai_report_out:
        print(report)

generate_report_btn.on_click(lambda _: _generate(force=False))
regenerate_report_btn.on_click(lambda _: _generate(force=True))
translate_btn.on_click(on_translate_click)


In [ ]:
# --- Layout ---
controls = widgets.VBox([
    study_dropdown, series_dropdown,
    widgets.HBox([slice_slider, zoom_slider]),
    window_slider,
])

labels_panel = widgets.VBox([widgets.HTML("<b>Ground-truth / editable labels (12)</b>")] + list(label_widgets.values()))

report_panel = widgets.VBox([
    widgets.HTML("<b>Original competition report</b>"), report_original_out,
    translate_btn,
    widgets.HTML("<b>English translation</b>"), report_translated_out,
])

ai_panel = widgets.VBox([
    widgets.HTML("<b>AI structured report (mri-interpreter)</b>"),
    widgets.HBox([generate_report_btn, regenerate_report_btn]),
    ai_report_out,
])

top = widgets.HBox([
    widgets.VBox([controls, image_out, meta_out]),
    widgets.VBox([labels_panel]),
])

display(top)
display(widgets.HBox([report_panel, ai_panel]))

# Kick off with the first study
on_study_change({"new": study_dropdown.value})
render()
